In [ ]:
import pandas as pd

In [ ]:
import os
import re
import time

import gspread
import openpyxl
from google.oauth2.credentials import Credentials


def read_tbl():
    df = pd.read_excel("../data/input/D2 Migration Schedule.xlsx")
    return df


def setup_gspread_client():
    """Initialize gspread client with OAuth credentials"""
    SCOPES = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive.readonly",
    ]

    # Load the OAuth token
    import json

    with open("token.json") as f:
        token_data = json.load(f)

    # Create credentials from the token
    creds = Credentials(
        token=token_data.get("token"),
        refresh_token=token_data.get("refresh_token"),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=token_data.get("client_id"),
        client_secret=token_data.get("client_secret"),
        scopes=SCOPES,
    )

    return gspread.authorize(creds)


def normalize_agency_name(agency_name):
    """
    Normalize agency name for comparison:
    - Convert to lowercase
    - Replace underscores with spaces
    - Strip whitespace
    """
    if agency_name:
        return agency_name.lower().replace("_", " ").strip()
    return None


def extract_agency_name(filename):
    """
    Extract agency name from filename format:
    autofolio_1.1.0_output--Agency Name--date_time
    or
    autofolio_1.1.0_output--Agency_Name--date_time
    Returns the normalized agency name or None if pattern doesn't match
    """
    # Remove .csv extension if present
    filename = filename.replace(".csv", "")

    # Pattern: autofolio_*_output--AGENCY_NAME--date_time
    # Agency name can have spaces or underscores
    pattern = r"autofolio_[\d.]+_output--(.+?)--\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}"
    match = re.match(pattern, filename)

    if match:
        agency = match.group(1)
        return normalize_agency_name(agency)
    return None


def get_existing_agencies(input_dir):
    """
    Get set of agency names that already exist in the input directory
    """
    existing_agencies = set()
    failed_to_parse = []

    if os.path.exists(input_dir):
        for filename in os.listdir(input_dir):
            if filename.endswith(".csv"):
                agency = extract_agency_name(filename)
                if agency:
                    existing_agencies.add(agency)
                else:
                    failed_to_parse.append(filename)

    print(f"Found {len(existing_agencies)} existing agencies in {input_dir}")

    if failed_to_parse:
        print(f"\nWARNING: Failed to parse {len(failed_to_parse)} files:")
        # Show first 10 examples
        for f in failed_to_parse[:10]:
            print(f"  - {f}")
        if len(failed_to_parse) > 10:
            print(f"  ... and {len(failed_to_parse) - 10} more")

    return existing_agencies


def extract_hyperlinks_from_excel(excel_path, sheet_name, column_name):
    """Extract hyperlinks from Excel file"""
    workbook = openpyxl.load_workbook(excel_path)

    # Try to find the sheet
    if sheet_name in workbook.sheetnames:
        worksheet = workbook[sheet_name]
    else:
        worksheet = workbook.active

    # Find the column
    headers = [cell.value for cell in worksheet[1]]
    if column_name not in headers:
        raise ValueError(f"Column '{column_name}' not found. Available columns: {headers}")

    col_index = headers.index(column_name) + 1

    hyperlinks = []
    for row in range(2, worksheet.max_row + 1):
        cell = worksheet.cell(row, col_index)
        if cell.value and cell.hyperlink:
            hyperlinks.append(
                {"display_text": cell.value, "url": cell.hyperlink.target, "row": row}
            )
        elif cell.value:
            print(f"Warning: Row {row} has value but no hyperlink: {cell.value}")

    return hyperlinks


def download_google_sheet_csv(client, sheet_url, output_filename):
    """Download Google Sheet as CSV using authenticated access"""
    try:
        # Extract sheet ID from URL
        sheet_id = sheet_url.split("/d/")[1].split("/")[0]

        # Open the spreadsheet with authentication
        spreadsheet = client.open_by_key(sheet_id)

        # Get the first worksheet
        worksheet = spreadsheet.sheet1

        # Get all values
        data = worksheet.get_all_values()

        # Convert to DataFrame and save as CSV
        if data:
            df = pd.DataFrame(data[1:], columns=data[0])
            df.to_csv(output_filename, index=False)
            return True
        else:
            print("    ✗ No data found in sheet")
            return False

    except Exception as e:
        print(f"    ✗ Error: {str(e)}")
        return False


def main():
    """Main function to process all autofolio sheets"""

    excel_path = "../data/input/D2 Migration Schedule.xlsx"
    sheet_name = "Migration Sheets"
    column_name = "Autofolio Sheet"
    input_dir = "../data/input/"
    output_dir = "../data/output/"

    os.makedirs(output_dir, exist_ok=True)

    # Get existing agencies to skip
    print("Checking for existing agencies...")
    existing_agencies = get_existing_agencies(input_dir)

    print("\nSetting up Google Sheets authentication...")
    client = setup_gspread_client()

    print("Extracting hyperlinks from Excel file...")
    hyperlinks = extract_hyperlinks_from_excel(excel_path, sheet_name, column_name)

    print(f"\n✓ Found {len(hyperlinks)} hyperlinks")

    # Filter out hyperlinks that already exist
    to_download = []
    skipped = []

    for item in hyperlinks:
        display_text = item["display_text"]
        agency = extract_agency_name(display_text)

        if agency and agency in existing_agencies:
            skipped.append({"agency": agency, "display_text": display_text})
        else:
            to_download.append(item)

    print(f"✓ Skipping {len(skipped)} already downloaded agencies")
    print(f"✓ Will download {len(to_download)} new agencies")
    print(f"\nStarting downloads to '{output_dir}/' directory...\n")

    success_count = 0
    for idx, item in enumerate(to_download):
        display_text = item["display_text"]
        # Clean the display text to use as filename
        filename = (
            display_text.replace(" ", "_").replace("/", "-").replace("\\", "-").replace(":", "-")
        )
        filepath = f"{output_dir}/{filename}.csv"

        print(f"[{idx + 1}/{len(to_download)}] {display_text}")
        if download_google_sheet_csv(client, item["url"], filepath):
            print(f"    ✓ Saved to: {filepath}")
            success_count += 1

        time.sleep(3)

    print(f"\n{'=' * 60}")
    print("Download complete!")
    print(f"Skipped (already exist): {len(skipped)}")
    print(f"Successfully downloaded: {success_count}/{len(to_download)} new sheets")
    print(f"Files saved to: {output_dir}/")
    print(f"{'=' * 60}")


main()